# Smart Travel

## Data Exploration

In this notebook, we will explore our initial dataset containing European travel destinations.

Goal:
- Load the dataset
- Understand its structure
- Prepare it for enrichment

In [38]:
import pandas as pd
import requests
import time

destinations = pd.read_csv("../data/raw/european_destinations.csv")
destinations.head()

,destination_name,country
0,Kefalonia,Greece
1,Santorini,Greece
2,Funchal,Portugal
3,Mallorca,Spain
4,Sardinia,Italy


In [39]:
url = "https://nominatim.openstreetmap.org/search"

headers = {
    "User-Agent": "smart-travel-project"
}

In [40]:
coordinates = []

for index, row in destinations.iterrows():
    query = row["destination_name"] + ", " + row["country"]

    params = {
        "q": query,
        "format": "json",
        "limit": 1
    }

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    if len(data) > 0:
        first_result = data[0]
        latitude = first_result["lat"]
        longitude = first_result["lon"]
    else:
        latitude = None
        longitude = None

    coordinates.append({
        "destination_name": row["destination_name"],
        "country": row["country"],
        "latitude": latitude,
        "longitude": longitude
    })

coordinates_df = pd.DataFrame(coordinates)

coordinates_df["latitude"] = coordinates_df["latitude"].astype(float)
coordinates_df["longitude"] = coordinates_df["longitude"].astype(float)

coordinates_df.head()

,destination_name,country,latitude,longitude
0,Kefalonia,Greece,38.266032,20.537390
1,Santorini,Greece,36.407111,25.456664
2,Funchal,Portugal,32.649650,-16.908678
3,Mallorca,Spain,39.613432,2.882919
4,Sardinia,Italy,40.091281,9.030577


In [41]:
import pandas as pd
import requests
import time

In [42]:
coordinates_df.to_csv("../data/processed/destinations_with_coordinates.csv", index=False)

In [43]:
def get_seasonal_weather(latitude, longitude):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2023-01-01",
        "end_date": "2025-12-31",
        "daily": ["temperature_2m_mean", "precipitation_sum"],
        "timezone": "auto"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    weather = pd.DataFrame(data["daily"])
    weather["time"] = pd.to_datetime(weather["time"])
    weather["month"] = weather["time"].dt.month
    
    seasons = {
        "winter": [12, 1, 2],
        "spring": [3, 4, 5],
        "summer": [6, 7, 8],
        "autumn": [9, 10, 11]
    }
    
    result = {}
    
    for season, months in seasons.items():
        season_data = weather[weather["month"].isin(months)]
        
        result[f"{season}_avg_temp"] = season_data["temperature_2m_mean"].mean()
        result[f"{season}_avg_daily_rain"] = season_data["precipitation_sum"].mean()
    
    return result

In [44]:
weather_data = []

for index, row in coordinates_df.iterrows():
    seasonal_weather = get_seasonal_weather(row["latitude"], row["longitude"])
    
    weather_data.append({
        "destination_name": row["destination_name"],
        "country": row["country"],
        **seasonal_weather
    })
    
    time.sleep(1)

weather_df = pd.DataFrame(weather_data)
weather_df.head()

,destination_name,country,winter_avg_temp,winter_avg_daily_rain,spring_avg_temp,spring_avg_daily_rain,summer_avg_temp,summer_avg_daily_rain,autumn_avg_temp,autumn_avg_daily_rain
0,Kefalonia,Greece,6.699631,3.812915,11.742391,2.291667,23.701812,0.411232,15.149084,3.795604
1,Santorini,Greece,14.540590,1.463100,17.621377,0.725000,26.657246,0.111957,21.900000,0.444689
2,Funchal,Portugal,16.246863,2.987085,17.537319,2.750362,22.426812,1.764855,20.833700,2.695971
3,Mallorca,Spain,10.953506,1.621771,15.683333,1.576087,25.518841,0.823913,18.926374,1.720513
4,Sardinia,Italy,7.380443,3.364576,11.784420,3.485145,23.142029,0.634783,15.512454,2.516484


In [45]:
destinations_enriched = coordinates_df.merge(
    weather_df,
    on=["destination_name", "country"],
    how="left"
)

numeric_columns = destinations_enriched.select_dtypes(include="number").columns

destinations_enriched[numeric_columns] = destinations_enriched[numeric_columns].round(2)

destinations_enriched.head()

,destination_name,country,latitude,longitude,winter_avg_temp,winter_avg_daily_rain,spring_avg_temp,spring_avg_daily_rain,summer_avg_temp,summer_avg_daily_rain,autumn_avg_temp,autumn_avg_daily_rain
0,Kefalonia,Greece,38.27,20.54,6.70,3.81,11.74,2.29,23.70,0.41,15.15,3.80
1,Santorini,Greece,36.41,25.46,14.54,1.46,17.62,0.73,26.66,0.11,21.90,0.44
2,Funchal,Portugal,32.65,-16.91,16.25,2.99,17.54,2.75,22.43,1.76,20.83,2.70
3,Mallorca,Spain,39.61,2.88,10.95,1.62,15.68,1.58,25.52,0.82,18.93,1.72
4,Sardinia,Italy,40.09,9.03,7.38,3.36,11.78,3.49,23.14,0.63,15.51,2.52


In [46]:
destinations_enriched.to_csv("../data/processed/destinations_enriched_weather.csv", index=False)